In [5]:
import numpy as np
import pandas as pd
import zarr
import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

TRAIN_PATH = Path("/Users/ishan/Desktop/biohub_kaggle/data/train")
_ZC = {} 

In [6]:
class Viewer:

    def __init__(
        self,
        vol_id: str,
        timestep: int = 0,
        verbose: bool = False,
    ):

        self.vol_id = vol_id
        self.verbose = verbose

        # Annotations cover every timestep, so load them once.
        self.nodes, self.edges = self.read_geff(TRAIN_PATH / (vol_id + ".geff"))
        self._build_track_colors()

        # Open the image volume (T, Z, Y, X) and cache the handle.
        self.zarr = self._open_zarr(vol_id)
        self.n_t, self.n_z = self.zarr.shape[0], self.zarr.shape[1]

        self.set_timestep(timestep)   # loads self.volume (t) and self.volume_tp1 (t+1)

        if self.verbose:
            print(f"Loaded {vol_id}: T={self.n_t} Z={self.n_z} "
                  f"nodes={len(self.nodes)} tracks={self.nodes['track'].nunique()}")

    def check(self):
        return self.volume, self.volume_tp1, self.nodes, self.edges

    def _open_zarr(self, vol_id):

        "Open the multiscale array '0' with shape (T, Z, Y, X), cached."

        key = str(TRAIN_PATH / (vol_id + ".zarr"))

        if key not in _ZC:
            _ZC[key] = zarr.open(key, mode="r")["0"]

        return _ZC[key]

    def set_timestep(self, t):

        "Load the (Z, Y, X) volume at t and its immediate successor t+1."

        self.t = int(t)
        self.volume = np.asarray(self.zarr[self.t])
        self.volume_tp1 = np.asarray(self.zarr[min(self.t + 1, self.n_t - 1)])

    def read_geff(self, geff_path):

        "Read the annotations stored as the respective geff directory."

        from geff import read as geff_read

        obj = geff_read(str(geff_path), backend="networkx")
        g = obj[0]
        nodes = pd.DataFrame([{"node_id": int(i), "t": int(round(float(d["t"]))),
                               "z": float(d["z"]), "y": float(d["y"]), "x": float(d["x"])}
                              for i, d in g.nodes(data=True)])
        nodes = nodes.sort_values(["t", "node_id"]).reset_index(drop=True)
        edges = pd.DataFrame([{"source_id": int(u), "target_id": int(v)} for u, v in g.edges()],
                             columns=["source_id", "target_id"])
        return nodes, edges

    def _build_track_colors(self):

        """Give every node a track id = connected component over the edges, so a
        source_id -> target_id chain (a cell followed across time) shares one color."""

        g = nx.Graph()
        g.add_nodes_from(self.nodes["node_id"].tolist())
        g.add_edges_from(self.edges[["source_id", "target_id"]].itertuples(index=False, name=None))

        node2track = {}
        for k, comp in enumerate(nx.connected_components(g)):
            for nid in comp:
                node2track[nid] = k

        self.nodes["track"] = self.nodes["node_id"].map(node2track)
        n_tracks = max(int(self.nodes["track"].max()) + 1, 1)
        cmap = plt.get_cmap("tab20", n_tracks)

        self.track_color = {k: cmap(k) for k in range(n_tracks)}
        self.node_color = {int(r.node_id): self.track_color[int(r.track)]
                           for r in self.nodes.itertuples()}

    def track_z(self, track_id, t):

        "z-index of the given track's node at timestep t, or None if it has no node there."

        s = self.nodes[(self.nodes["track"] == track_id) & (self.nodes["t"] == t)]
        return int(round(s["z"].iloc[0])) if len(s) else None

    def draw_volumes(self, ax1, ax2, z1, z2):

        ax1.imshow(self.volume[z1], cmap="gray")
        ax2.imshow(self.volume_tp1[z2], cmap="gray")

    def draw_nodes(self, ax, t, z_slice, follow=None):

        "Overlay the nodes at timestep t; color = track, filled if on this z-slice."

        sub = self.nodes[self.nodes["t"] == t]

        for r in sub.itertuples():

            color = self.node_color[int(r.node_id)]
            on_slice = int(round(r.z)) == int(z_slice)

            ax.scatter(r.x, r.y,
                       s=140 if on_slice else 60,
                       facecolors=color if on_slice else "none",
                       edgecolors=color, linewidths=2, marker="o", zorder=3)

            if follow is not None and int(r.track) == follow:   # ring the followed cell
                ax.scatter(r.x, r.y, s=340, facecolors="none",
                           edgecolors=color, linewidths=2, marker="o", zorder=4)

            ax.annotate(f"trk{int(r.track)}·z{int(round(r.z))}", (r.x, r.y),
                        color=color, fontsize=8, xytext=(5, 5),
                        textcoords="offset points", zorder=3)

    def update_view(self, t, z1, z2, follow=None):

        "Render volume[t] @ z1 next to volume[t+1] @ z2, with tracked nodes overlaid."

        self.set_timestep(t)

        zt = sorted(int(round(v)) for v in self.nodes.loc[self.nodes.t == t, "z"])
        zt1 = sorted(int(round(v)) for v in self.nodes.loc[self.nodes.t == t + 1, "z"])
        print(f"nodes  t={t}: z={zt or '—'}   |   t+1={t + 1}: z={zt1 or '—'}")

        fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))

        self.draw_volumes(ax1, ax2, z1, z2)
        self.draw_nodes(ax1, t, z1, follow)
        self.draw_nodes(ax2, t + 1, z2, follow)

        ax1.set_title(f"t = {t}   (z = {z1})")
        ax2.set_title(f"t = {t + 1}   (z = {z2})")
        ax1.axis("off")
        ax2.axis("off")
        fig.tight_layout()
        plt.show()

    def view(self):

        """
        Interactive widget to select the timestep (t) and the two z-slices
        (z1 for t, z2 for t+1).

        The 'follow' dropdown locks onto one track: while it is set, moving the t
        slider auto-snaps z1 and z2 to that cell's z-index at t and t+1, so you can
        track a single cell through depth across timesteps (the followed node is
        ringed). Pick 'Off' to drive z1/z2 manually again.
        """

        ts = sorted(int(v) for v in self.nodes["t"].unique())
        ts_set = set(ts)
        t0 = next((t for t in ts if (t + 1) in ts_set), (ts[0] if ts else 0))

        def _z_at(t):
            s = self.nodes[self.nodes["t"] == t]
            return int(round(s["z"].iloc[0])) if len(s) else 0

        track_ids = sorted(int(v) for v in self.nodes["track"].unique())

        track_dd = widgets.Dropdown(
            options=[("Off (manual z)", None)] + [(f"track {k}", k) for k in track_ids],
            value=None, description="follow")
        
        t_slider = widgets.IntSlider(value=t0, min=0, max=self.n_t - 2, step=1, description="t")
        z1_slider = widgets.IntSlider(value=_z_at(t0), min=0, max=self.n_z - 1, step=1, description="z1")
        z2_slider = widgets.IntSlider(value=_z_at(t0 + 1), min=0, max=self.n_z - 1, step=1, description="z2")

        def _sync_z(*_):

            "When a track is selected, drive z1/z2 from that track's centroid z."

            tk = track_dd.value

            if tk is None:
                z1_slider.disabled = z2_slider.disabled = False
                return
            
            z1_slider.disabled = z2_slider.disabled = True   # z is now track-driven
            z1 = self.track_z(tk, t_slider.value)
            z2 = self.track_z(tk, t_slider.value + 1)

            if z1 is not None:
                z1_slider.value = z1
            if z2 is not None:
                z2_slider.value = z2

        t_slider.observe(_sync_z, names="value")
        track_dd.observe(_sync_z, names="value")

        out = widgets.interactive_output(
            self.update_view,
            {"t": t_slider, "z1": z1_slider, "z2": z2_slider, "follow": track_dd},
        )

        ui = widgets.VBox([widgets.HBox([track_dd, t_slider]),
                           widgets.HBox([z1_slider, z2_slider])])
        display(ui, out)

In [7]:
viewer = Viewer("6bba_2540cd90", timestep=0, verbose=True)
viewer.view()

Loaded 6bba_2540cd90: T=100 Z=64 nodes=529 tracks=6


Output()

In [31]:
nodes, edges = viewer.read_geff(TRAIN_PATH / "6bba_2540cd90.geff")

In [32]:
g = nx.Graph()
g.add_nodes_from(nodes["node_id"].tolist())
g.add_edges_from(edges[["source_id", "target_id"]].itertuples(index=False, name=None))

In [ ]:
node2track = {}

for k, comp in enumerate(nx.connected_components(g)):
    for nid in comp:
        node2track[nid] = k

{66000391, 22000145, 81000466, 44000277, 96000541, 15000097, 59000355, 37000237, 74000431, 8000049, 89000506, 52000320, 1000001, 30000194, 67000396, 82000471, 23000152, 45000282, 97000545, 16000104, 60000361, 38000244, 75000436, 9000056, 90000511, 53000325, 2000007, 31000200, 68000401, 83000476, 24000158, 46000288, 98000549, 61000366, 17000111, 76000441, 39000250, 10000063, 91000516, 54000330, 3000013, 32000206, 69000406, 84000481, 25000164, 47000294, 99000553, 62000371, 18000118, 77000446, 40000255, 11000070, 92000521, 55000335, 4000020, 33000212, 70000411, 85000486, 26000170, 48000300, 100000557, 63000376, 19000125, 78000451, 41000260, 12000077, 93000526, 56000340, 34000218, 5000028, 71000416, 86000491, 27000176, 49000305, 64000381, 20000132, 79000456, 42000266, 13000083, 94000531, 57000345, 35000224, 6000035, 72000421, 87000496, 28000182, 50000310, 65000386, 21000139, 80000461, 43000272, 95000536, 14000090, 58000350, 36000230, 7000042, 73000426, 88000501, 51000315, 29000188}
{660003

In [43]:
nodes["track"] = nodes["node_id"].map(node2track)

In [46]:
nodes["track"].unique()

array([0, 1, 2, 3, 4, 5])

In [47]:
nodes.groupby("track").get_group(1)

,node_id,t,z,y,x,track
1,1000002,0,9.0,199.0,212.0,1
6,2000008,1,9.0,197.0,210.0,1
11,3000014,2,10.0,192.0,210.0,1
16,4000021,3,10.0,193.0,206.0,1
22,5000029,4,10.0,193.0,204.0,1
...,...,...,...,...,...,...
510,96000542,95,38.0,242.0,116.0,1
514,97000546,96,38.0,245.0,112.0,1
518,98000550,97,38.0,245.0,109.0,1
522,99000554,98,38.0,247.0,108.0,1


In [40]:
nodes.groupby("t").get_group(1)

,node_id,t,z,y,x
5,2000007,1,7.0,178.0,67.0
6,2000008,1,9.0,197.0,210.0
7,2000010,1,15.0,133.0,182.0
8,2000011,1,15.0,26.0,22.0
9,2000012,1,22.0,7.0,206.0


In [41]:
nodes.groupby("t").get_group(2)

,node_id,t,z,y,x
10,3000013,2,7.0,176.0,65.0
11,3000014,2,10.0,192.0,210.0
12,3000016,2,15.0,131.0,180.0
13,3000017,2,15.0,24.0,22.0
14,3000018,2,23.0,5.0,205.0


In [42]:
nodes.groupby("t").get_group(3)

,node_id,t,z,y,x
15,4000020,3,8.0,174.0,62.0
16,4000021,3,10.0,193.0,206.0
17,4000023,3,14.0,146.0,253.0
18,4000024,3,16.0,127.0,177.0
19,4000025,3,16.0,22.0,23.0
20,4000026,3,23.0,4.0,206.0
